# Tech Challenge Fase 4 — LSTM para Previsão de Preços de Ações

**Pós Tech — Machine Learning Engineering**

Este notebook documenta todo o processo de desenvolvimento do modelo:

1. Coleta de dados via `yfinance`
2. Análise exploratória
3. Pré-processamento (normalização + janelamento)
4. Construção e treino da LSTM
5. Avaliação com MAE, RMSE e MAPE
6. Salvamento do modelo para deploy

## 1. Imports e configuração

In [ ]:
import os
import sys
import json
from pathlib import Path

# Garante que o diretório raiz do projeto esteja no PYTHONPATH
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import tensorflow as tf

from src.data_loader import fetch_stock_data, get_close_prices
from src.preprocessor import prepare_dataset, inverse_transform, save_scaler
from src.model import build_lstm_model, get_callbacks
from src.evaluate import compute_metrics, print_metrics

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

print('TensorFlow:', tf.__version__)
print('GPU disponível:', len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Coleta de dados

In [ ]:
SYMBOL = 'AAPL'
START_DATE = '2018-01-01'
END_DATE = '2024-07-20'

df = fetch_stock_data(SYMBOL, START_DATE, END_DATE)
print(f'Dados de {SYMBOL}: {len(df)} dias úteis')
df.head()

In [ ]:
df.describe()

## 3. Análise exploratória

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df.index, df['Close'], color='steelblue')
axes[0].set_title(f'{SYMBOL} — Preço de Fechamento Histórico')
axes[0].set_ylabel('Preço (USD)')

axes[1].plot(df.index, df['Volume'], color='gray', alpha=0.6)
axes[1].set_title('Volume Negociado')
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

In [ ]:
# Retornos diários
df['Returns'] = df['Close'].pct_change()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['Returns'].dropna().hist(bins=80, ax=axes[0], color='teal')
axes[0].set_title('Distribuição dos Retornos Diários')

df['Close'].rolling(30).std().plot(ax=axes[1], color='crimson')
axes[1].set_title('Volatilidade (desvio-padrão móvel 30d)')
plt.tight_layout()
plt.show()

print(f'Retorno médio diário: {df["Returns"].mean()*100:.3f}%')
print(f'Volatilidade anualizada: {df["Returns"].std()*np.sqrt(252)*100:.2f}%')

## 4. Pré-processamento

In [ ]:
WINDOW_SIZE = 60
TRAIN_RATIO = 0.8

closes = get_close_prices(df)
X_train, X_test, y_train, y_test, scaler = prepare_dataset(
    closes,
    window_size=WINDOW_SIZE,
    train_ratio=TRAIN_RATIO,
)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape} | y_test : {y_test.shape}')

## 5. Construção e treinamento da LSTM

In [ ]:
model = build_lstm_model(
    window_size=WINDOW_SIZE,
    lstm_units_1=64,
    lstm_units_2=64,
    dropout_rate=0.2,
    learning_rate=0.001,
)
model.summary()

In [ ]:
callbacks = list(get_callbacks(patience=10))

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    shuffle=False,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss (MSE)')
axes[0].legend()

axes[1].plot(history.history['mae'], label='train')
axes[1].plot(history.history['val_mae'], label='val')
axes[1].set_title('MAE')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Avaliação

In [ ]:
y_pred_scaled = model.predict(X_test).flatten()
y_pred = inverse_transform(y_pred_scaled, scaler)
y_true = inverse_transform(y_test, scaler)

metrics = compute_metrics(y_true, y_pred)
print_metrics(metrics)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(y_true, label='Real', color='steelblue')
plt.plot(y_pred, label='Previsto', color='crimson', alpha=0.8)
plt.title(f'{SYMBOL} — Real vs Previsto (conjunto de teste)')
plt.xlabel('Dias')
plt.ylabel('Preço (USD)')
plt.legend()
plt.show()

In [ ]:
errors = y_true - y_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(errors, color='purple')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Resíduos (real - previsto)')

axes[1].hist(errors, bins=40, color='purple', alpha=0.7)
axes[1].set_title('Distribuição dos Resíduos')
plt.tight_layout()
plt.show()

## 7. Salvamento dos artefatos

In [ ]:
from datetime import datetime

MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / 'lstm_model.keras'
scaler_path = MODELS_DIR / 'scaler.pkl'
metadata_path = MODELS_DIR / 'metadata.json'

model.save(model_path)
save_scaler(scaler, str(scaler_path))

metadata = {
    'symbol': SYMBOL,
    'start_date': START_DATE,
    'end_date': END_DATE,
    'window_size': WINDOW_SIZE,
    'lstm_units_1': 64,
    'lstm_units_2': 64,
    'dropout_rate': 0.2,
    'epochs_trained': len(history.history['loss']),
    'batch_size': 32,
    'learning_rate': 0.001,
    'train_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'metrics': metrics,
    'trained_at': datetime.utcnow().isoformat() + 'Z',
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'✅ Modelo salvo em: {model_path}')
print(f'✅ Scaler salvo em: {scaler_path}')
print(f'✅ Metadata salvo em: {metadata_path}')

## 8. Próximos passos

Com os artefatos salvos, a API FastAPI em `src/api/main.py` consegue carregá-los
automaticamente no startup. Para testar:

```bash
uvicorn src.api.main:app --reload
# Acesse http://localhost:8000/docs
```

Ou via Docker Compose com Prometheus + Grafana:

```bash
docker-compose up -d
```